
# 🛣️ AI-Based Pothole Detection & Road Condition Classification

**A GPU-accelerated computer vision pipeline for automated road-condition surveying.**

Built for Google Colab (free T4 GPU) — YOLOv8 · OpenCV · Gradio · Folium · Geopy

---

### What this notebook does

| Stage | Technology | Output |
|---|---|---|
| 1. Object Detection | YOLOv8 (Ultralytics) | Bounding boxes around potholes |
| 2. Severity Estimation | Bounding-box-area heuristic | Low / Medium / High Risk |
| 3. Geolocation | EXIF GPS → mock-GPS fallback | Latitude / Longitude |
| 4. Reverse Geocoding | Geopy + OpenStreetMap Nominatim | Human-readable street address |
| 5. Interactive Mapping | Folium | Pin-drop map with severity popup |
| 6. Reporting | Pandas → CSV | Downloadable survey log |
| 7. UI | Gradio (`gr.Blocks`, 3 tabs) | End-to-end web app, runs inline in Colab |

### Notebook structure

0. Environment check (GPU)
1. Install dependencies
2. Imports & global configuration
3. Load YOLOv8 model (with a transparent fallback path)
   - 3B. *(Optional)* Fine-tune YOLOv8 on your own pothole dataset
4. Core CV utilities — severity classification + baseline fallback detector
5. Geospatial utilities — GPS resolution (EXIF-first) + reverse geocoding
6. Folium interactive map builder
7. Reporting — detection log + CSV export
8. Master pipeline — orchestrates 1→7 into a single function
9. Gradio multi-tab web app
10. Notes, limitations & production roadmap

> **Engineering note:** Section 3 is written so the *entire* pipeline runs end-to-end
> even before you have a custom-trained model — a classical OpenCV heuristic stands
> in for detection until you drop in real YOLOv8 pothole weights (see Section 3B).
> This keeps the notebook demo-able on day one while leaving a clean upgrade path
> to production accuracy.



## 0. Environment Check

Confirm the T4 GPU is attached before doing anything expensive. In Colab:
`Runtime → Change runtime type → Hardware accelerator → GPU (T4)`.


In [3]:
# Quick hardware sanity check — run this first.
!nvidia-smi

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch version: 2.14.0+cpu
CUDA available: False
Running on CPU


'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.



## 1. Install Dependencies

`ultralytics` ships YOLOv8. `opencv-python-headless` avoids GUI-lib conflicts in
Colab's headless environment. Everything else backs the geospatial + UI layers.


In [16]:
%pip install -q ultralytics gradio folium geopy opencv-python-headless

print("✅ Dependencies installed.")

Note: you may need to restart the kernel to use updated packages.
✅ Dependencies installed.



## 2. Imports & Global Configuration

Centralizing every tunable constant here (severity thresholds, marker colors,
the default geocoding anchor point, file paths) means the rest of the notebook
never hardcodes a "magic number" — change behavior in one place.


In [5]:
import os
import cv2
import math
import random
import numpy as np
import pandas as pd
import torch
from datetime import datetime

from PIL import Image
from PIL.ExifTags import TAGS, GPSTAGS

import folium
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

import gradio as gr
from ultralytics import YOLO

print("✅ Imports OK.")


✅ Imports OK.


In [1]:
import sys
print(sys.executable)

import torch
print(torch.__version__)

c:\Users\kanig\OneDrive\Desktop\Pothole_Detection-main\Pothole_Detection-main\venv\Scripts\python.exe
2.14.0+cpu


In [2]:
import sys
print(sys.executable)

import torch
print("PyTorch version:", torch.__version__)

c:\Users\kanig\OneDrive\Desktop\Pothole_Detection-main\Pothole_Detection-main\venv\Scripts\python.exe
PyTorch version: 2.14.0+cpu


In [6]:
# ── Severity thresholds ─────────────────────────────────────────────────
# Expressed as (bounding-box area) / (total image area). Tune these against
# your own camera's field of view / mounting height for real deployments.
SEVERITY_THRESHOLDS = {"low": 0.02, "medium": 0.06}

# ── Visual encoding of severity, shared by OpenCV drawing + Folium markers ─
SEVERITY_COLORS_BGR = {                    # OpenCV uses BGR, not RGB
    "Low":           (0, 200, 0),
    "Medium":        (0, 165, 255),
    "High Risk":      (0, 0, 255),
}
SEVERITY_COLORS_HEX = {                    # for HTML popups
    "Low":            "#2ecc71",
    "Medium":         "#f39c12",
    "High Risk":       "#e74c3c",
    "None Detected":   "#3498db",
}
FOLIUM_ICON_COLOR = {                      # folium.Icon only accepts a fixed palette
    "Low":            "green",
    "Medium":         "orange",
    "High Risk":       "red",
    "None Detected":   "blue",
}
FOLIUM_ICON_SYMBOL = {                     # Bootstrap glyphicon names
    "Low":            "info-sign",
    "Medium":         "warning-sign",
    "High Risk":       "exclamation-sign",
    "None Detected":   "ok-sign",
}

# ── Mock-GPS anchor: Bengaluru city center — used only when an uploaded ───
# image has no EXIF GPS tags (true for almost all downloaded test images).
DEFAULT_CITY_LAT = 12.9716
DEFAULT_CITY_LON = 77.5946
MOCK_GPS_RADIUS_KM = 8.0

# ── Custom-trained pothole weights (see Section 3 / 3B) ────────────────────
MODEL_PATH = "/content/pothole_yolov8_best.pt"

# ── Working paths ───────────────────────────────────────────────────────
TMP_UPLOAD_PATH = "/content/_tmp_upload.jpg"
CSV_PATH = "/content/pothole_detection_report.csv"

print("✅ Config loaded.")


✅ Config loaded.



## 3. Load the YOLOv8 Model

**Production path:** point `MODEL_PATH` (set above) at weights you fine-tuned on a
pothole dataset (see Section 3B) — the pipeline will load and use them directly.

**Demo path:** if no custom weights exist yet, `USE_CUSTOM_MODEL` becomes `False`
and the pipeline transparently falls back to a classical OpenCV heuristic detector
(Section 4) so the whole notebook still runs end-to-end. This is intentional
engineering, not a bug — it means anyone can clone this notebook and get a working
demo immediately, then swap in a trained model without touching any other cell.


In [7]:
USE_CUSTOM_MODEL = os.path.exists(MODEL_PATH)

if USE_CUSTOM_MODEL:
    model = YOLO(MODEL_PATH)
    model.to("cuda" if torch.cuda.is_available() else "cpu")
    print(f"✅ Loaded custom pothole-trained YOLOv8 model from: {MODEL_PATH}")
else:
    model = None
    print("⚠️  No custom weights found at MODEL_PATH.")
    print("    Running in DEMO MODE: a classical CV heuristic (Section 4) will")
    print("    stand in for pothole detection so the pipeline still runs.")
    print("    → Train your own model in Section 3B, save weights to MODEL_PATH,")
    print("      then re-run this cell to switch to real YOLOv8 inference.")


⚠️  No custom weights found at MODEL_PATH.
    Running in DEMO MODE: a classical CV heuristic (Section 4) will
    stand in for pothole detection so the pipeline still runs.
    → Train your own model in Section 3B, save weights to MODEL_PATH,
      then re-run this cell to switch to real YOLOv8 inference.



### 3B. *(Optional)* Fine-Tune YOLOv8 on a Custom Pothole Dataset

Skip this section entirely if you already have `best.pt` weights. This is a
**template** — plug in your own [Roboflow](https://roboflow.com) API key and
project (Roboflow Universe hosts several public pothole-detection datasets you
can fork into your own workspace), then flip `RUN_TRAINING = True`.

Training a YOLOv8n checkpoint for ~50 epochs on a few hundred pothole images
comfortably fits in a Colab T4 session.


In [17]:
import os
import cv2
import numpy as np
import pandas as pd
from datetime import datetime

# ============================================================
# OPTIONAL YOLO IMPORT
# ============================================================

try:
    from ultralytics import YOLO
    YOLO_AVAILABLE = True
except ImportError:
    YOLO_AVAILABLE = False


# ============================================================
# CONFIGURATION
# ============================================================

MODEL_PATH = "pothole_yolov8_best.pt"
CSV_PATH = "pothole_detection_report.csv"

CONFIDENCE_THRESHOLD = 0.25

# YOLO model will be used if this file exists
USE_YOLO = os.path.exists(MODEL_PATH) and YOLO_AVAILABLE


# ============================================================
# GLOBAL VARIABLES
# ============================================================

model = None

if USE_YOLO:
    try:
        print("Loading YOLO model...")
        model = YOLO(MODEL_PATH)
        print("YOLO model loaded successfully.")
    except Exception as e:
        print("Could not load YOLO model.")
        print("Reason:", e)
        model = None
        USE_YOLO = False
else:
    print("Custom YOLO weights not found.")
    print("Running in OpenCV DEMO MODE.")


# ============================================================
# DETECTION LOG
# ============================================================

columns = [
    "Date",
    "Time",
    "Detection",
    "Severity",
    "Area",
    "Confidence"
]

if os.path.exists(CSV_PATH):
    try:
        detection_log = pd.read_csv(CSV_PATH)
    except Exception:
        detection_log = pd.DataFrame(columns=columns)
else:
    detection_log = pd.DataFrame(columns=columns)


# ============================================================
# SEVERITY CLASSIFICATION
# ============================================================

def calculate_severity(area, image_area):
    """
    Estimate pothole severity using detected area
    compared with total image area.
    """

    if image_area <= 0:
        return "Unknown"

    percentage = (area / image_area) * 100

    if percentage < 2:
        return "Low"
    elif percentage < 6:
        return "Medium"
    else:
        return "High"


# ============================================================
# OPENCV FALLBACK DETECTOR
# ============================================================

def opencv_pothole_detection(image):
    """
    Simple classical computer vision fallback.

    This is NOT a trained AI model.
    It uses dark-region and contour analysis.
    """

    output = image.copy()

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Blur image
    blur = cv2.GaussianBlur(gray, (7, 7), 0)

    # Threshold dark regions
    _, threshold = cv2.threshold(
        blur,
        70,
        255,
        cv2.THRESH_BINARY_INV
    )

    # Morphological operations
    kernel = np.ones((5, 5), np.uint8)

    threshold = cv2.morphologyEx(
        threshold,
        cv2.MORPH_CLOSE,
        kernel
    )

    threshold = cv2.morphologyEx(
        threshold,
        cv2.MORPH_OPEN,
        kernel
    )

    contours, _ = cv2.findContours(
        threshold,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    image_area = image.shape[0] * image.shape[1]

    detections = []

    for contour in contours:

        area = cv2.contourArea(contour)

        # Ignore very small regions
        if area < 500:
            continue

        x, y, w, h = cv2.boundingRect(contour)

        # Ignore extremely small objects
        if w < 30 or h < 30:
            continue

        # Avoid detecting huge regions
        if area > image_area * 0.5:
            continue

        severity = calculate_severity(
            area,
            image_area
        )

        detections.append({
            "x": x,
            "y": y,
            "w": w,
            "h": h,
            "area": area,
            "severity": severity,
            "confidence": 0.50
        })

    return detections


# ============================================================
# YOLO DETECTOR
# ============================================================

def yolo_pothole_detection(image):

    detections = []

    if model is None:
        return detections

    try:

        results = model.predict(
            source=image,
            conf=CONFIDENCE_THRESHOLD,
            verbose=False,
            device="cpu"
        )

        image_area = image.shape[0] * image.shape[1]

        for result in results:

            if result.boxes is None:
                continue

            for box in result.boxes:

                xyxy = box.xyxy[0].cpu().numpy()

                x1, y1, x2, y2 = map(
                    int,
                    xyxy
                )

                confidence = float(
                    box.conf[0].cpu().numpy()
                )

                x1 = max(0, x1)
                y1 = max(0, y1)
                x2 = min(image.shape[1], x2)
                y2 = min(image.shape[0], y2)

                width = max(0, x2 - x1)
                height = max(0, y2 - y1)

                area = width * height

                severity = calculate_severity(
                    area,
                    image_area
                )

                detections.append({
                    "x": x1,
                    "y": y1,
                    "w": width,
                    "h": height,
                    "area": area,
                    "severity": severity,
                    "confidence": confidence
                })

    except Exception as e:

        print("YOLO detection error:", e)

    return detections


# ============================================================
# DRAW DETECTIONS
# ============================================================

def draw_detections(image, detections):

    output = image.copy()

    for i, detection in enumerate(detections, start=1):

        x = detection["x"]
        y = detection["y"]
        w = detection["w"]
        h = detection["h"]

        severity = detection["severity"]
        confidence = detection["confidence"]

        # Different display color based on severity
        if severity == "High":
            color = (0, 0, 255)

        elif severity == "Medium":
            color = (0, 165, 255)

        else:
            color = (0, 255, 0)

        # Bounding box
        cv2.rectangle(
            output,
            (x, y),
            (x + w, y + h),
            color,
            3
        )

        label = (
            f"Pothole {i} | "
            f"{severity} | "
            f"{confidence:.2f}"
        )

        # Text background
        text_size = cv2.getTextSize(
            label,
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            2
        )[0]

        cv2.rectangle(
            output,
            (x, max(0, y - 35)),
            (
                x + text_size[0] + 10,
                y
            ),
            color,
            -1
        )

        cv2.putText(
            output,
            label,
            (x + 5, max(20, y - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 255),
            2
        )

    return output


# ============================================================
# SAVE REPORT
# ============================================================

def save_report(detections):

    global detection_log

    now = datetime.now()

    rows = []

    for detection in detections:

        rows.append({
            "Date": now.strftime("%Y-%m-%d"),
            "Time": now.strftime("%H:%M:%S"),
            "Detection": "Pothole",
            "Severity": detection["severity"],
            "Area": round(detection["area"], 2),
            "Confidence": round(
                detection["confidence"],
                3
            )
        })

    if rows:

        new_data = pd.DataFrame(rows)

        detection_log = pd.concat(
            [
                detection_log,
                new_data
            ],
            ignore_index=True
        )

        detection_log.to_csv(
            CSV_PATH,
            index=False
        )


# ============================================================
# MAIN DETECTION FUNCTION
# ============================================================

def detect_potholes(image):

    global detection_log

    if image is None:
        return None, detection_log

    # Gradio provides RGB image
    rgb_image = np.array(image)

    # Convert RGB → BGR for OpenCV
    bgr_image = cv2.cvtColor(
        rgb_image,
        cv2.COLOR_RGB2BGR
    )

    # ========================================================
    # SELECT DETECTOR
    # ========================================================

    if USE_YOLO and model is not None:

        print("Using YOLOv8 detector.")

        detections = yolo_pothole_detection(
            bgr_image
        )

        detector_name = "YOLOv8"

    else:

        print("Using OpenCV fallback detector.")

        detections = opencv_pothole_detection(
            bgr_image
        )

        detector_name = "OpenCV Demo"


    # ========================================================
    # DRAW RESULTS
    # ========================================================

    output = draw_detections(
        bgr_image,
        detections
    )

    # ========================================================
    # SAVE REPORT
    # ========================================================

    save_report(detections)

    # ========================================================
    # ADD INFORMATION TO IMAGE
    # ========================================================

    cv2.putText(
        output,
        f"Detector: {detector_name}",
        (20, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )

    cv2.putText(
        output,
        f"Potholes detected: {len(detections)}",
        (20, 70),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )

    # Convert BGR → RGB
    output_rgb = cv2.cvtColor(
        output,
        cv2.COLOR_BGR2RGB
    )

    return output_rgb, detection_log


# ============================================================
# GRADIO INTERFACE
# ============================================================

def create_interface():

    import gradio as gr

    with gr.Blocks(
        title="AI Pothole Detection System"
    ) as demo:

        gr.Markdown(
            """
            # 🚧 AI-Based Pothole Detection System

            Upload a road image to detect potholes,
            estimate their severity and generate a detection report.
            """
        )

        gr.Markdown(
            """
            **Detection modes:**

            - YOLOv8 → used when `pothole_yolov8_best.pt` exists
            - OpenCV → automatic fallback when YOLO weights are unavailable
            """
        )

        with gr.Row():

            input_image = gr.Image(
                type="numpy",
                label="Upload Road Image"
            )

            output_image = gr.Image(
                type="numpy",
                label="Detection Result"
            )

        detect_button = gr.Button(
            "🔍 Detect Potholes",
            variant="primary"
        )

        detection_table = gr.Dataframe(
            headers=columns,
            value=detection_log,
            label="Detection Log",
            interactive=False
        )

        detect_button.click(
            fn=detect_potholes,
            inputs=input_image,
            outputs=[
                output_image,
                detection_table
            ]
        )

        gr.Markdown(
            """
            ### Severity

            🟢 **Low** – small pothole  
            🟠 **Medium** – moderate pothole  
            🔴 **High** – large pothole
            """
        )

    return demo


# ============================================================
# RUN APPLICATION
# ============================================================

if __name__ == "__main__":

    print("=" * 60)
    print("AI POTHOLE DETECTION SYSTEM")
    print("=" * 60)

    print("Python application started.")

    if USE_YOLO:
        print("Detection mode: YOLOv8")
    else:
        print("Detection mode: OpenCV DEMO")

    print(
        f"Report file: {os.path.abspath(CSV_PATH)}"
    )

    demo = create_interface()

    demo.launch(
        server_name="127.0.0.1",
        server_port=7860,
        share=False
    )

Custom YOLO weights not found.
Running in OpenCV DEMO MODE.
AI POTHOLE DETECTION SYSTEM
Python application started.
Detection mode: OpenCV DEMO
Report file: c:\Users\kanig\OneDrive\Desktop\Pothole_Detection-main\Pothole_Detection-main\notebooks\pothole_detection_report.csv
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.



## 4. Core CV Utilities

Two building blocks:

1. **`classify_severity`** — turns a bounding box's area (as a fraction of the
   full image) into a Low / Medium / High Risk label.
2. **`detect_potholes_classical`** — the demo-mode fallback detector. It looks
   for dark, irregular blobs on the road surface via adaptive thresholding and
   contour filtering. It is deliberately simple and is **not** a substitute for
   a trained model — it exists purely so the pipeline is runnable without a
   dataset. Swap in real YOLOv8 weights (Section 3) for production accuracy.


In [8]:
def classify_severity(area_ratio: float) -> str:
    """
    Classify pothole severity from bounding-box area as a fraction of the
    total image area. Larger apparent area (closer / bigger pothole) → higher risk.
    """
    if area_ratio < SEVERITY_THRESHOLDS["low"]:
        return "Low"
    elif area_ratio < SEVERITY_THRESHOLDS["medium"]:
        return "Medium"
    else:
        return "High Risk"


def detect_potholes_classical(img_bgr: np.ndarray):
    """
    Baseline heuristic pothole-candidate detector, used ONLY when no custom
    YOLOv8 weights are loaded (USE_CUSTOM_MODEL == False).

    Approach: potholes usually appear as darker, irregularly-shaped patches
    against a more uniform road surface, so we:
      1. Blur to suppress texture noise.
      2. Adaptive-threshold to isolate locally-darker regions.
      3. Morphologically close small gaps so a pothole reads as one blob.
      4. Filter contours by area and aspect ratio to reject shadows / noise.

    Returns a list of (x1, y1, x2, y2, pseudo_confidence) tuples — the same
    shape the YOLOv8 branch produces, so downstream code doesn't care which
    detector ran.
    """
    h, w = img_bgr.shape[:2]
    img_area = h * w

    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (7, 7), 0)
    thresh = cv2.adaptiveThreshold(
        blurred, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY_INV, 25, 8
    )
    kernel = np.ones((5, 5), np.uint8)
    cleaned = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)

    contours, _ = cv2.findContours(cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    candidates = []
    for c in contours:
        area = cv2.contourArea(c)
        area_ratio = area / img_area
        if area_ratio < 0.004 or area_ratio > 0.35:
            continue  # too small = noise, too large = probably shadow/whole road
        x, y, bw, bh = cv2.boundingRect(c)
        aspect = bw / float(bh)
        if aspect < 0.25 or aspect > 4.0:
            continue  # reject very thin slivers (cracks, lane lines, shadows)
        pseudo_conf = min(0.95, 0.4 + area_ratio)  # bigger dark blob → higher heuristic confidence
        candidates.append((x, y, x + bw, y + bh, pseudo_conf))

    # Cap to the strongest 8 candidates to keep the demo output readable
    candidates = sorted(candidates, key=lambda b: b[4], reverse=True)[:8]
    return candidates


print("✅ CV utility functions ready.")


✅ CV utility functions ready.



## 5. Geospatial Utilities

**GPS resolution is EXIF-first:** we always try to read real GPS tags out of
the image metadata before falling back to a mock coordinate. Most test images
(screenshots, downloaded stock photos) strip EXIF entirely, so the mock
generator uses uniform-disk sampling around a real city center (defaults to
Bengaluru) so points look like plausible street locations rather than clustering
unrealistically near the center.

**Reverse geocoding** uses Geopy's Nominatim client (OpenStreetMap). Nominatim's
usage policy caps free requests at ~1/sec, so we wrap it in a `RateLimiter`.


In [9]:
def get_exif_gps(image_path: str):
    """Extract (lat, lon) from an image's EXIF GPS tags, or None if absent/unreadable."""
    try:
        img = Image.open(image_path)
        exif_data = img._getexif()
        if not exif_data:
            return None

        gps_info = {}
        for tag, value in exif_data.items():
            if TAGS.get(tag, tag) == "GPSInfo":
                for gps_tag in value:
                    gps_info[GPSTAGS.get(gps_tag, gps_tag)] = value[gps_tag]

        if not gps_info or "GPSLatitude" not in gps_info:
            return None

        def to_degrees(dms):
            d, m, s = dms
            return float(d) + float(m) / 60.0 + float(s) / 3600.0

        lat = to_degrees(gps_info["GPSLatitude"])
        if gps_info.get("GPSLatitudeRef", "N") != "N":
            lat = -lat

        lon = to_degrees(gps_info["GPSLongitude"])
        if gps_info.get("GPSLongitudeRef", "E") != "E":
            lon = -lon

        return round(lat, 6), round(lon, 6)
    except Exception:
        return None


def generate_mock_gps(center_lat=DEFAULT_CITY_LAT, center_lon=DEFAULT_CITY_LON,
                       radius_km=MOCK_GPS_RADIUS_KM):
    """
    Generate a realistic random GPS coordinate within `radius_km` of a city
    center, using uniform sampling over a disk (not a square) so points don't
    cluster unrealistically near the center point.
    """
    radius_deg = radius_km / 111.0  # ≈111 km per degree of latitude
    u, v = random.random(), random.random()
    dist = radius_deg * math.sqrt(u)
    angle = 2 * math.pi * v
    dlat = dist * math.cos(angle)
    dlon = dist * math.sin(angle) / math.cos(math.radians(center_lat))
    return round(center_lat + dlat, 6), round(center_lon + dlon, 6)


def resolve_gps(image_path: str):
    """EXIF GPS if present, otherwise a mock coordinate. Returns ((lat, lon), source_label)."""
    exif_gps = get_exif_gps(image_path)
    if exif_gps:
        return exif_gps, "EXIF (real)"
    return generate_mock_gps(), "Mock (simulated)"


print("✅ GPS utilities ready.")


✅ GPS utilities ready.


In [10]:
# A descriptive, unique user_agent is required by Nominatim's usage policy.
geolocator = Nominatim(user_agent="ai_pothole_detector_colab_v1")

# Nominatim's free tier asks for ~1 request/second — RateLimiter enforces that
# automatically and retries transient failures instead of crashing the app.
_reverse_geocode_rl = RateLimiter(
    geolocator.reverse, min_delay_seconds=1, max_retries=2, error_wait_seconds=2
)


def reverse_geocode(lat: float, lon: float) -> str:
    """Turn coordinates into a human-readable street address via Nominatim."""
    try:
        location = _reverse_geocode_rl((lat, lon), exactly_one=True, timeout=10)
        return location.address if location else "Address not found for these coordinates."
    except (GeocoderTimedOut, GeocoderServiceError) as e:
        return f"Geocoding service unavailable ({e}). Showing coordinates only."
    except Exception as e:
        return f"Geocoding error: {e}"


print("✅ Reverse-geocoding function ready.")


✅ Reverse-geocoding function ready.



## 6. Folium Interactive Map Builder

Builds a Folium map centered on the detection point, with a color-coded marker
(green / orange / red / blue by severity) and a popup showing severity, address,
and coordinates. `map_to_html` renders it to an HTML string suitable for a
Gradio `gr.HTML` component.


In [11]:
def build_map(lat: float, lon: float, severity: str, address: str, num_potholes: int) -> folium.Map:
    """Create a Folium map with a single severity-colored marker for this detection."""
    fmap = folium.Map(location=[lat, lon], zoom_start=16, tiles="OpenStreetMap",
                       width="100%", height="480px")

    popup_html = f"""
    <div style="font-family: Arial, sans-serif; font-size: 13px; width: 230px;">
      <h4 style="margin:0 0 6px 0; color:{SEVERITY_COLORS_HEX.get(severity, '#333')};">
        ⚠ {severity}
      </h4>
      <b>Potholes detected:</b> {num_potholes}<br>
      <b>Address:</b> {address}<br>
      <b>Coordinates:</b> {lat}, {lon}
    </div>
    """

    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(popup_html, max_width=260),
        tooltip=f"{severity} — click for details",
        icon=folium.Icon(
            color=FOLIUM_ICON_COLOR.get(severity, "blue"),
            icon=FOLIUM_ICON_SYMBOL.get(severity, "info-sign"),
        ),
    ).add_to(fmap)

    # Soft radius circle so the pin reads clearly even when zoomed out
    folium.Circle(
        location=[lat, lon], radius=40,
        color=SEVERITY_COLORS_HEX.get(severity, "#3498db"),
        fill=True, fill_opacity=0.15, weight=1,
    ).add_to(fmap)

    return fmap


def map_to_html(fmap: folium.Map) -> str:
    """Render a Folium map to an HTML string embeddable in a Gradio gr.HTML component."""
    return f'<div style="width:100%;">{fmap._repr_html_()}</div>'


def default_map_html() -> str:
    """Placeholder map shown before the first detection has run."""
    fmap = folium.Map(location=[DEFAULT_CITY_LAT, DEFAULT_CITY_LON], zoom_start=12,
                       width="100%", height="480px")
    return map_to_html(fmap)


print("✅ Map builder ready.")


✅ Map builder ready.



## 7. Reporting — Detection Log & CSV Export

Every analysis run (including "road looks clear" results) is appended to an
in-memory Pandas DataFrame, which doubles as the survey log shown in the UI and
the source for the downloadable CSV report.


In [13]:
REPORT_COLUMNS = [
    "Detection_ID", "Date", "Address", "Latitude", "Longitude",
    "Severity", "Potholes_Count", "GPS_Source",
]
detection_log = pd.DataFrame(columns=REPORT_COLUMNS)


def log_detection(address, lat, lon, severity, count, gps_source):
    """Append one detection record to the running survey log."""
    global detection_log
    new_row = {
        "Detection_ID": len(detection_log) + 1,
        "Date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Address": address,
        "Latitude": lat,
        "Longitude": lon,
        "Severity": severity,
        "Potholes_Count": count,
        "GPS_Source": gps_source,
    }
    detection_log = pd.concat([detection_log, pd.DataFrame([new_row])], ignore_index=True)
    return detection_log


def export_csv() -> str:
    """Write the current survey log to disk and return its path for Gradio's File component."""
    detection_log.to_csv(CSV_PATH, index=False)
    return CSV_PATH


print("✅ Reporting utilities ready.")


✅ Reporting utilities ready.



## 8. Master Detection Pipeline

Ties every section above into one function: detect → estimate severity →
resolve GPS → reverse-geocode → draw boxes → build map → log → export CSV.
This single function is what the Gradio UI calls on each "Analyze" click.


In [14]:
def run_pipeline(pil_image):
    """
    Full pipeline for one uploaded road image.

    Returns a 6-tuple matching the Gradio output components:
    (original_image, annotated_image, info_text, map_html, csv_path, detection_log_df)
    """
    if pil_image is None:
        return (None, None, "⚠️ Please upload an image first.",
                default_map_html(), None, detection_log)

    # Save a temp copy — lets us attempt real EXIF GPS extraction before falling
    # back to mock coordinates (Gradio's in-memory PIL image alone loses EXIF).
    pil_image.save(TMP_UPLOAD_PATH)

    img_bgr = cv2.cvtColor(np.array(pil_image.convert("RGB")), cv2.COLOR_RGB2BGR)
    h, w = img_bgr.shape[:2]
    img_area = h * w

    # ── 1. Detection ────────────────────────────────────────────────────
    if USE_CUSTOM_MODEL:
        results = model.predict(img_bgr, verbose=False)[0]
        boxes = [
            (*map(int, b.xyxy[0].tolist()), float(b.conf[0]))
            for b in results.boxes
        ]
    else:
        boxes = detect_potholes_classical(img_bgr)

    # ── 2. Severity per box + draw annotations ──────────────────────────
    annotated = img_bgr.copy()
    per_box_severity = []
    for (x1, y1, x2, y2, conf) in boxes:
        area_ratio = ((x2 - x1) * (y2 - y1)) / img_area
        sev = classify_severity(area_ratio)
        per_box_severity.append(sev)

        color = SEVERITY_COLORS_BGR[sev]
        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 3)
        label = f"{sev} ({conf * 100:.0f}%)"
        cv2.putText(annotated, label, (x1, max(20, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    # Overall road-segment severity = worst individual detection (safety-first)
    if per_box_severity:
        rank = {"Low": 0, "Medium": 1, "High Risk": 2}
        overall_severity = max(per_box_severity, key=lambda s: rank[s])
    else:
        overall_severity = "None Detected"

    # ── 3. Geolocation + reverse geocoding ──────────────────────────────
    (lat, lon), gps_source = resolve_gps(TMP_UPLOAD_PATH)
    address = reverse_geocode(lat, lon)

    # ── 4. Build report text ────────────────────────────────────────────
    info_text = "\n".join([
        f"🕳️  Potholes detected: {len(boxes)}",
        f"🚦  Overall Road Severity: {overall_severity}",
        f"📍  Location: {address}",
        f"🌐  Coordinates: {lat}, {lon}   (GPS source: {gps_source})",
        f"🧠  Detector: {'Custom YOLOv8' if USE_CUSTOM_MODEL else 'Classical CV (demo fallback)'}",
    ])

    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    # ── 5. Map ───────────────────────────────────────────────────────────
    fmap = build_map(lat, lon, overall_severity, address, len(boxes))
    map_html = map_to_html(fmap)

    # ── 6. Log + CSV ─────────────────────────────────────────────────────
    log_detection(address, lat, lon, overall_severity, len(boxes), gps_source)
    csv_path = export_csv()

    return pil_image, Image.fromarray(annotated_rgb), info_text, map_html, csv_path, detection_log


print("✅ Master pipeline ready.")


✅ Master pipeline ready.



## 9. Gradio Multi-Tab Web App

Three tabs sharing one "Analyze" button:

- **Tab 1 — Detection:** upload an image, see the original vs. annotated result,
  and a text summary of severity + location.
- **Tab 2 — Interactive Map:** the Folium map for the most recent detection.
- **Tab 3 — Data Export:** the full survey log as a table, plus a CSV download.

All three tabs update from a single click because Gradio lets one button's
outputs span components defined in different tabs.


In [15]:
with gr.Blocks(title="AI Pothole Detection & Road Condition Classification") as demo:
    gr.Markdown("# 🛣️ AI-Based Pothole Detection & Road Condition Classification")
    gr.Markdown(
        "Upload a road image to detect potholes, estimate severity, geotag the "
        "location, and log it to a downloadable survey report."
    )

    with gr.Tabs():
        with gr.Tab("1️⃣ Detection"):
            with gr.Row():
                inp_image = gr.Image(type="pil", label="Upload Road Image")
                out_original = gr.Image(label="Original")
                out_annotated = gr.Image(label="Detected Potholes")
            out_info = gr.Textbox(label="Severity & Location Report", lines=6)
            btn_detect = gr.Button("🔍 Analyze Road Image", variant="primary")

        with gr.Tab("2️⃣ Interactive Map"):
            out_map = gr.HTML(value=default_map_html(), label="Detection Map")

        with gr.Tab("3️⃣ Data Export"):
            out_table = gr.Dataframe(value=detection_log, label="Detection Log", interactive=False)
            out_csv = gr.File(label="Download CSV Report")

    btn_detect.click(
        fn=run_pipeline,
        inputs=[inp_image],
        outputs=[out_original, out_annotated, out_info, out_map, out_csv, out_table],
    )

print("✅ Gradio app built. Run the next cell to launch it.")


✅ Gradio app built. Run the next cell to launch it.


In [18]:
# share=True gives you a public link too (handy for testing from a phone camera).
# debug=True streams errors into this cell's output instead of failing silently.
demo.launch(share=True, debug=True)


Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


Using OpenCV fallback detector.
Keyboard interruption in main thread... closing server.



## 10. Notes, Limitations & Production Roadmap

**Current limitations (by design, for a portable demo notebook):**
- Without custom weights, detection runs on a classical CV heuristic, not a
  trained model — real-world accuracy will be modest until you train on actual
  pothole imagery (Section 3B).
- Mock GPS is only used when EXIF GPS is absent — most consumer/dashcam photos
  strip EXIF, so treat mock coordinates as illustrative, not survey-grade.
- The in-memory `detection_log` resets when the Colab runtime restarts; for a
  persistent deployment, write to a database or cloud bucket instead of a local CSV.

**Suggested next steps for a production system:**
1. Train YOLOv8 (or YOLOv8-seg for pixel-accurate area) on a labeled pothole
   dataset — Section 3B has a ready-made Roboflow template.
2. Replace bounding-box-area severity with a depth/area estimate calibrated
   against known camera height and focal length for real-world pothole size.
3. Feed real GPS from a phone/dashcam rig (or vehicle telemetry) instead of
   relying on EXIF, and batch-process video frames instead of single images.
4. Swap the local CSV for a proper backend (PostGIS / TimescaleDB) so pins
   accumulate into a city-wide road-condition dashboard over time.
5. Export the trained model via `model.export(format="onnx")` or TensorRT for
   low-latency inference on edge devices (e.g., a Jetson mounted in a vehicle).
